# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule in plain words:

Pages that experienced a position tier drop between Week 1 (March 1–7) and Week 2 (March 8–15) will be prioritized for refresh review. Among these declining pages, those with higher search visibility will receive a higher priority score because they will affect more people

Reason code: Ranking decline

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

import duckdb
from huggingface_hub import login

login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

file_path

'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

In [3]:
page_performance_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- Week 1 (March 1-6)
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_clicks ELSE 0 END) AS week1_clicks,
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_impressions ELSE 0 END) AS week1_impressions,
    AVG(CASE WHEN report_date <= '2026-03-07' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week1_avg_position,

    -- Week 2 (March 8-15)
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS week2_clicks,
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS week2_impressions,
    AVG(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week2_avg_position,

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

page_performance = con.sql(page_performance_query).df()
page_performance.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position
0,client_9958f0a7ae1df715,content_810cf06597918291,0.0,57.0,8.451437,0.0,64.0,9.001880
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,0.0,14.0,5.642857,1.0,69.0,8.114490
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,38.0,4797.0,4.726033,67.0,4970.0,4.720103
3,client_9958f0a7ae1df715,content_278030b007943b07,1.0,50.0,6.495074,4.0,124.0,6.441830
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,10.0,2039.0,6.721713,5.0,1552.0,7.190038


In [4]:
import numpy as np
import pandas as pd

def position_tier(pos):
    if pd.isna(pos) or pos == 0:
      return np.nan

    if pos <= 3:
        return 1
    elif pos <= 10:
        return 2
    elif pos <= 20:
        return 3
    elif pos <= 50:
        return 4
    else:
        return 5

page_performance["week1_position_tier"] = page_performance["week1_avg_position"].apply(position_tier)
page_performance["week2_position_tier"] = page_performance["week2_avg_position"].apply(position_tier)
page_performance["tier_change"] = page_performance["week2_position_tier"] - page_performance["week1_position_tier"]

page_performance[["client_hash_id", "content_hash_id", "week1_avg_position", "week2_avg_position",
        "week1_position_tier", "week2_position_tier", "tier_change"]].head(10)

,client_hash_id,content_hash_id,week1_avg_position,week2_avg_position,week1_position_tier,week2_position_tier,tier_change
0,client_9958f0a7ae1df715,content_810cf06597918291,8.451437,9.001880,2.0,2.0,0.0
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,5.642857,8.114490,2.0,2.0,0.0
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,4.726033,4.720103,2.0,2.0,0.0
3,client_9958f0a7ae1df715,content_278030b007943b07,6.495074,6.441830,2.0,2.0,0.0
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,6.721713,7.190038,2.0,2.0,0.0
5,client_9958f0a7ae1df715,content_347fbafb77d3ae37,17.366270,20.259072,3.0,4.0,1.0
6,client_9958f0a7ae1df715,content_15bd72d24e0a0b08,9.458036,19.182488,2.0,3.0,1.0
7,client_9958f0a7ae1df715,content_6f4cc70af7be346e,10.843288,9.400871,3.0,2.0,-1.0
8,client_9958f0a7ae1df715,content_c5c8a9160d8b64ae,27.323016,20.817708,4.0,4.0,0.0
9,client_9958f0a7ae1df715,content_85e499a6915dc41e,5.666667,7.875000,2.0,2.0,0.0


In [5]:
print(page_performance[["tier_change"]].isna().sum())
print(len(page_performance))
print(f"tier_change missing: {34852/len(page_performance):.1%}")

tier_change    49593
dtype: int64
63856
tier_change missing: 54.6%


In [6]:
def position_change_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change < 0:
        return "improved"
    elif change == 0:
        return "stable"
    else:
        return "declined"

page_performance["position_change"] = page_performance["tier_change"].apply(position_change_bucket)

In [7]:
position_signal_table = (
    page_performance[page_performance["position_change"].notna()]
    ["position_change"]
    .value_counts()
    .reset_index()
)

position_signal_table.columns = ["position_change", "n"]

position_signal_table

,position_change,n
0,stable,10036
1,improved,2300
2,declined,1927


Position change verdict: MIXED

A meaningful subset of the pages with active search data are declining in ranking (13%), indicating that ranking decline is a reasonable signal for prioritising pages for refresh review. However, the short window means that some of the tier changes could be due to rank turbulence rather than content decay. This is shown by 16% of the pages with active search data experiencing an increase in ranking.

In [8]:
def visibility_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change <= 8:
        return "low"
    elif change > 8 and change <= 459:
        return "medium"
    else:
        return "high"

page_performance["visibility_bucket"] = page_performance["week2_impressions"].apply(visibility_bucket)

In [9]:
visibility_table = (
    page_performance[page_performance["visibility_bucket"].notna()]
    ["visibility_bucket"]
    .value_counts()
    .reset_index()
)

visibility_table.columns = ["Visibility bucket", "n"]

print(visibility_table)

  Visibility bucket      n
0               low  35905
1            medium  20004
2              high   7947


Visibility verdict: MIXED

By itself, the visibility of each page cannot tell us if a page is declining or not. However, when used alongside position change it can be used to help us decide which pages to prioritise for review, as pages with high visibility have greater search exposure, meaning they will benefit more from being reviewed for refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
page_performance["visibility_score"] = page_performance["visibility_bucket"].map({
    "low": 1,
    "medium": 2,
    "high": 4
})

page_performance

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,week1_position_tier,week2_position_tier,tier_change,position_change,visibility_bucket,visibility_score
0,client_9958f0a7ae1df715,content_810cf06597918291,0.0,57.0,8.451437,0.0,64.0,9.001880,2.0,2.0,0.0,stable,medium,2
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,0.0,14.0,5.642857,1.0,69.0,8.114490,2.0,2.0,0.0,stable,medium,2
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,38.0,4797.0,4.726033,67.0,4970.0,4.720103,2.0,2.0,0.0,stable,high,4
3,client_9958f0a7ae1df715,content_278030b007943b07,1.0,50.0,6.495074,4.0,124.0,6.441830,2.0,2.0,0.0,stable,medium,2
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,10.0,2039.0,6.721713,5.0,1552.0,7.190038,2.0,2.0,0.0,stable,high,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_32058aa8a2e4f4fb,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1
63852,client_20259bd6705d81d4,content_526be944d717ab76,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1
63853,client_20259bd6705d81d4,content_20c613ae83bad2ca,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1
63854,client_20259bd6705d81d4,content_d4f53cf222510f09,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1


In [11]:
def calculate_score(row):
    if row["tier_change"] > 0:
        return row["tier_change"] * row["visibility_score"]
    else:
        return 0

page_performance["page_score"] = page_performance.apply(calculate_score, axis=1)



page_performance

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,week1_position_tier,week2_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score
0,client_9958f0a7ae1df715,content_810cf06597918291,0.0,57.0,8.451437,0.0,64.0,9.001880,2.0,2.0,0.0,stable,medium,2,0.0
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,0.0,14.0,5.642857,1.0,69.0,8.114490,2.0,2.0,0.0,stable,medium,2,0.0
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,38.0,4797.0,4.726033,67.0,4970.0,4.720103,2.0,2.0,0.0,stable,high,4,0.0
3,client_9958f0a7ae1df715,content_278030b007943b07,1.0,50.0,6.495074,4.0,124.0,6.441830,2.0,2.0,0.0,stable,medium,2,0.0
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,10.0,2039.0,6.721713,5.0,1552.0,7.190038,2.0,2.0,0.0,stable,high,4,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_32058aa8a2e4f4fb,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1,0.0
63852,client_20259bd6705d81d4,content_526be944d717ab76,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1,0.0
63853,client_20259bd6705d81d4,content_20c613ae83bad2ca,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1,0.0
63854,client_20259bd6705d81d4,content_d4f53cf222510f09,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1,0.0


In [12]:
page_performance[page_performance["page_score"] > 0][
    ["tier_change", "visibility_bucket", "visibility_score", "page_score"]
].head(10)

,tier_change,visibility_bucket,visibility_score,page_score
5,1.0,medium,2,2.0
6,1.0,medium,2,2.0
11,1.0,low,1,1.0
15,1.0,medium,2,2.0
19,1.0,medium,2,2.0
29,1.0,medium,2,2.0
30,1.0,medium,2,2.0
32,1.0,medium,2,2.0
38,1.0,medium,2,2.0
41,1.0,medium,2,2.0


In [13]:
page_performance["reason_code"] = "ranking_decline"
page_performance["action"] = "review for refresh"

#Tie-breaker
page_performance = page_performance.sort_values(
    by=["page_score", "content_hash_id"],
    ascending=[False, True]
).reset_index(drop=True)


page_performance

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,week1_position_tier,week2_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score,reason_code,action
0,client_20259bd6705d81d4,content_3091099eccc89076,1.0,29.0,1.413793,1.0,1022.0,31.843771,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
1,client_20259bd6705d81d4,content_35c904673f215f0b,1.0,130.0,1.600000,2.0,1896.0,36.378582,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
2,client_20259bd6705d81d4,content_66296153ae89bce6,1.0,48.0,2.208333,1.0,1066.0,36.430725,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
3,client_23a62021009f63c4,content_694a059af51e30c7,1.0,264.0,2.972664,4.0,3345.0,24.386915,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
4,client_ff644d8251367cbb,content_e9a90a47969568a0,1.0,79.0,1.341772,1.0,583.0,34.469756,1.0,4.0,3.0,declined,high,4,12.0,ranking_decline,review for refresh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_3197e6291363b4db,content_fffc042bb9e93d49,0.0,1.0,NaN,1.0,8.0,2.625000,NaN,1.0,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
63852,client_73cda7b4e4f265ea,content_fffc19fa9a6e2ae2,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
63853,client_fef1a8f436438636,content_fffc7f2781221803,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
63854,client_23a62021009f63c4,content_ffffb3ccfb4a3482,0.0,0.0,NaN,0.0,10.0,4.100000,NaN,2.0,NaN,NaN,medium,2,0.0,ranking_decline,review for refresh


In [14]:
ranked_queue = page_performance[page_performance["page_score"] > 0][["client_hash_id", "content_hash_id", "page_score", "reason_code","action"]]
ranked_queue = ranked_queue.sort_values("page_score", ascending=False)

#Tie-breaker
ranked_queue = ranked_queue.sort_values(
    by=["page_score", "content_hash_id"],
    ascending=[False, True]
).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked_queue.head(20)

,client_hash_id,content_hash_id,page_score,reason_code,action
0,client_20259bd6705d81d4,content_3091099eccc89076,12.0,ranking_decline,review for refresh
1,client_20259bd6705d81d4,content_35c904673f215f0b,12.0,ranking_decline,review for refresh
2,client_20259bd6705d81d4,content_66296153ae89bce6,12.0,ranking_decline,review for refresh
3,client_23a62021009f63c4,content_694a059af51e30c7,12.0,ranking_decline,review for refresh
4,client_ff644d8251367cbb,content_e9a90a47969568a0,12.0,ranking_decline,review for refresh
5,client_20259bd6705d81d4,content_f5086cd82e84e011,12.0,ranking_decline,review for refresh
6,client_23a62021009f63c4,content_0198cdd755fa7a52,8.0,ranking_decline,review for refresh
7,client_23a62021009f63c4,content_0ca05069ff9c8f43,8.0,ranking_decline,review for refresh
8,client_23a62021009f63c4,content_112aee06ac44905a,8.0,ranking_decline,review for refresh
9,client_23a62021009f63c4,content_309c2ccc855ac350,8.0,ranking_decline,review for refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Out of the top 20 pages in the queue, the first 6 rows have the highest page score possible. This is due to having the largest combination of ranking decline and visibility. I confirmed this by observing the ranking decline and visibility level of each page, meaning I am confident that the pages are ranked according to my baseline rule.However, the rankings presented in the queue could be wrong due to the decline in ranking being caused by reasons unrelated to the content of each page. For example, it could be a seasonal issue or a technical issue with the webpage.

The last 14 pages below have a lower page score. This is an indicator of either a smaller ranking decline with higher visibility or a middling ranking decline with middling visibility. I confirmed this by observing the ranking decline and visibility level of each page, meaning I am confident that the pages are ranked according to my baseline rule. However, these rankings could still be wrong due to similar reasons as the higher ranking pages such as seasonality or technical issues.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = ranked_queue.head(20)
top_20

,client_hash_id,content_hash_id,page_score,reason_code,action
0,client_20259bd6705d81d4,content_3091099eccc89076,12.0,ranking_decline,review for refresh
1,client_20259bd6705d81d4,content_35c904673f215f0b,12.0,ranking_decline,review for refresh
2,client_20259bd6705d81d4,content_66296153ae89bce6,12.0,ranking_decline,review for refresh
3,client_23a62021009f63c4,content_694a059af51e30c7,12.0,ranking_decline,review for refresh
4,client_ff644d8251367cbb,content_e9a90a47969568a0,12.0,ranking_decline,review for refresh
5,client_20259bd6705d81d4,content_f5086cd82e84e011,12.0,ranking_decline,review for refresh
6,client_23a62021009f63c4,content_0198cdd755fa7a52,8.0,ranking_decline,review for refresh
7,client_23a62021009f63c4,content_0ca05069ff9c8f43,8.0,ranking_decline,review for refresh
8,client_23a62021009f63c4,content_112aee06ac44905a,8.0,ranking_decline,review for refresh
9,client_23a62021009f63c4,content_309c2ccc855ac350,8.0,ranking_decline,review for refresh


In [16]:
# Full table used to confirm ranked queue results.
# Exact ordering differs from ranked queue for each score.
# For example, the pages with a page score of 12 are in a different order

page_performance = page_performance.sort_values("page_score", ascending=False)
page_performance = page_performance.reset_index(drop=True)


page_performance[['client_hash_id', 'content_hash_id', 'tier_change', 'visibility_score', 'page_score',	'reason_code',	'action']].head(20)

,client_hash_id,content_hash_id,tier_change,visibility_score,page_score,reason_code,action
0,client_20259bd6705d81d4,content_3091099eccc89076,3.0,4,12.0,ranking_decline,review for refresh
1,client_20259bd6705d81d4,content_35c904673f215f0b,3.0,4,12.0,ranking_decline,review for refresh
2,client_20259bd6705d81d4,content_66296153ae89bce6,3.0,4,12.0,ranking_decline,review for refresh
3,client_23a62021009f63c4,content_694a059af51e30c7,3.0,4,12.0,ranking_decline,review for refresh
4,client_ff644d8251367cbb,content_e9a90a47969568a0,3.0,4,12.0,ranking_decline,review for refresh
5,client_20259bd6705d81d4,content_f5086cd82e84e011,3.0,4,12.0,ranking_decline,review for refresh
6,client_23a62021009f63c4,content_925e8207de232ece,2.0,4,8.0,ranking_decline,review for refresh
7,client_23a62021009f63c4,content_979799969a9011ff,2.0,4,8.0,ranking_decline,review for refresh
8,client_20259bd6705d81d4,content_a4ecaa6f93a7e2bd,2.0,4,8.0,ranking_decline,review for refresh
9,client_23a62021009f63c4,content_a69a749eb06db926,2.0,4,8.0,ranking_decline,review for refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

While my top 20 results appear to rank the pages effectively, with the highest ranked pages having the highest tier change, lower down in the queue a page with high visibility but a small ranking decline could potentially be prioritised higher than pages with greater ranking decline but low visibility.
These pages with small decline may not be declining enough to justify needing a review, potentially causing the rankings to be incorrect.

My baseline rule only uses historical data available in the dataset to determine the page's ranking and visibility level in the first half of March 2026, nefore my decision point 15th March 2026. It does not use any labels or later data, so there is no data leakage.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.